[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/name-generation-rnn.ipynb)

# Character-Level Name Generation with RNN

This notebook trains a character-level RNN to generate names, then analyzes how many generated names are novel versus memorized from the training data.

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import numpy as np
import wandb
from pytorch_lightning.loggers import WandbLogger

# Import shared utilities from local package
from aiml_notebooks import CharacterTokenizer, NamesDataset, collate_fn, create_dataset, create_dataloaders

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'seed': 42,                      # Random seed for reproducibility
    'train_split': 0.9,              # Fraction of data for training (rest is validation)
    'batch_size': 128,               # Number of names per training batch
    
    # Model
    'embedding_dim': 64,             # Size of character embedding vectors
    'hidden_size': 256,              # Number of units in RNN hidden layers
    'num_layers': 2,                 # Number of stacked RNN layers
    'dropout': 0.2,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 30,                # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Generation
    'max_length': 15,                # Maximum characters in generated names
    'temperature': 0.8,              # Sampling randomness (lower=conservative, higher=creative)
    'sample_size': 20,               # Number of example names to generate
    'novelty_sample_size': 100,      # Number of names for novelty analysis
    
    # Weights & Biases
    'wandb_project': 'name-generation-rnn',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Next, we'll define all hyperparameters in a CONFIG dictionary and set random seeds for reproducibility.

In [ ]:
# Papermill parameters (injected by sweep runner)
# These values will override CONFIG when running sweeps
embedding_dim = CONFIG['embedding_dim']
hidden_size = CONFIG['hidden_size']
num_layers = CONFIG['num_layers']
dropout = CONFIG['dropout']
learning_rate = CONFIG['learning_rate']
batch_size = CONFIG['batch_size']
max_epochs = CONFIG['max_epochs']
temperature = CONFIG['temperature']
wandb_project = CONFIG['wandb_project']
wandb_run_name = CONFIG['wandb_run_name']

# Update CONFIG with papermill parameters
CONFIG.update({
    'embedding_dim': embedding_dim,
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'max_epochs': max_epochs,
    'temperature': temperature,
    'wandb_project': wandb_project,
    'wandb_run_name': wandb_run_name,
})

print(f"Running with config: {CONFIG}")

These papermill parameters allow hyperparameter sweeps to override the default CONFIG values.

In [ ]:
# Create dataset using the factory (handles data loading, tokenization, and splitting)
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id="names",
    splits=[CONFIG['train_split'], 1 - CONFIG['train_split']]
)

# Extract the tokenizer from the full dataset for later use
tokenizer = full_dataset.tokenizer

print(f"Total names: {len(full_dataset)}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Tokenizer: {tokenizer}")

Now we'll use the dataset factory to download the names, create the tokenizer, and split the data.

In [ ]:
# Create data loaders using the factory
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['batch_size']
)

Now we'll use the dataloader factory to create batched, shuffled loaders for training and validation.

In [ ]:
# Define the character-level RNN model with embedding, RNN layers, and generation method
class NameGeneratorRNN(L.LightningModule):
    def __init__(self, vocab_size: int, embedding_dim: int = 64, hidden_size: int = 256,
                 num_layers: int = 2, dropout: float = 0.2, learning_rate: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        rnn_out, hidden = self.rnn(embedded, hidden)
        rnn_out = self.dropout(rnn_out)
        logits = self.fc(rnn_out)
        return logits, hidden
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('val_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    @torch.no_grad()
    def generate(self, tokenizer, max_length=20, temperature=1.0, num_samples=1):
        """Generate names using the tokenizer."""
        self.eval()
        generated_names = []
        
        for _ in range(num_samples):
            current_idx = tokenizer.get_special_token_idx()
            name_chars = []
            hidden = None
            
            for _ in range(max_length):
                x = torch.tensor([[current_idx]], dtype=torch.long, device=self.device)
                logits, hidden = self(x, hidden)
                probs = F.softmax(logits[0, -1] / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
                next_char = tokenizer.decode_char(next_idx)
                
                if tokenizer.is_special_token(next_char):
                    break
                
                name_chars.append(next_char)
                current_idx = next_idx
            
            generated_names.append(''.join(name_chars))
        
        return generated_names

# Initialize the model with CONFIG hyperparameters
model = NameGeneratorRNN(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
)

Now we'll define our RNN model with embedding, recurrent, and output layers, plus a generation method.

In [ ]:
# Initialize Weights & Biases logger
# Check if we're running inside an existing W&B run (e.g., from a sweep)
if wandb.run is not None:
    print(f"Using existing W&B run: {wandb.run.name}")
    wandb_logger = None  # Use existing run instead of creating new logger
else:
    # Create new W&B run
    wandb_logger = WandbLogger(
        project=CONFIG['wandb_project'],
        name=CONFIG['wandb_run_name'],
        config=CONFIG,
    )
    print(f"Created W&B run: {wandb_logger.experiment.name}")

# Train the model using PyTorch Lightning trainer
trainer = L.Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='auto',
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=CONFIG['log_every_n_steps'],
    logger=wandb_logger,
)
trainer.fit(model, train_loader, val_loader)

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Generate sample names
generated = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['sample_size']
)
generated = [name.capitalize() for name in generated]
print(", ".join(generated))

# Analyze novelty (new vs existing names)
print("\n" + "="*50)
print("NOVELTY ANALYSIS")
print("="*50)

sample = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['novelty_sample_size']
)
sample = [name for name in sample if name]

# Get original names from the full dataset for comparison
original_names_set = set(full_dataset.names)
new_names = [name for name in sample if name not in original_names_set]
existing_names = [name for name in sample if name in original_names_set]

# Calculate metrics
novelty_pct = len(new_names) / len(sample) * 100
uniqueness_pct = len(set(sample)) / len(sample) * 100

print(f"Total: {len(sample)}, Unique: {len(set(sample))}")
print(f"✨ NEW: {len(new_names)} ({novelty_pct:.1f}%)")
print(f"♻️  EXISTING: {len(existing_names)} ({len(existing_names)/len(sample)*100:.1f}%)")

print(f"\nNEW: {', '.join([n.capitalize() for n in new_names[:10]])}")
print(f"EXISTING: {', '.join([n.capitalize() for n in existing_names[:10]])}")

# Log to W&B
wandb.log({
    'novelty_percentage': novelty_pct,
    'uniqueness_percentage': uniqueness_pct,
    'total_generated': len(sample),
    'new_names_count': len(new_names),
    'memorized_names_count': len(existing_names),
})

# Log sample names as a table
wandb.log({
    'sample_new_names': wandb.Table(
        columns=['name'],
        data=[[n.capitalize()] for n in new_names[:20]]
    )
})

Finally, we'll generate sample names and analyze how many are novel versus memorized from training.